In [1]:
%gui qt6

# Plotting stuff
import pyqtgraph as pg
import matplotlib.pyplot as plt
import cmasher as cmr

# Science stuff
import numpy as np
import pandas as pd
from spectres import spectres

# Astropy stuff
from astropy.io import fits
from astropy.table import Table
import astropy.units as u
from astropy.units.quantity import Quantity
from astropy.io.ascii import read as ascii_read

# General stuff
import logging
import sys

# zHunter stuff
from zhunter.initialize import DIRS
from zhunter import io
from zhunter.misc import generate_fake_1D_spectrum
from zhunter.spectrum import OneDSpectrum

# Colors
from zhunter.colors import COLORS
color_style = 'kraken17'
colors = COLORS[color_style]


# Logging
log = logging.getLogger(__name__)
logging.basicConfig(stream=sys.stdout, level=logging.DEBUG,
    format="%(asctime)s.%(msecs)03d | %(levelname)-8s | %(funcName)s - %(filename)s:%(lineno)d : %(message)s",

                   )
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("PIL").setLevel(logging.WARNING)

# Create Spectrum Object

## Load data

In [2]:
# From a filename
spec1 = OneDSpectrum()
spec1.load_from_file(fname='../../data/test_input_files/FORS_1D.fits')

# can also load data directly but data must be quantity
spec2 = OneDSpectrum()
spec2.load_from_data(*generate_fake_1D_spectrum())

# Unit-less flux
spec3 = OneDSpectrum()
spec3.load_from_file(fname='../../data/test_input_files/GRB050730_UVES_1D.txt.gz')

2024-11-07 15:04:43,095.095 | DEBUG    | __init__ - spectrum.py:226 : Initializing OneDSpectrum instance called: 
2024-11-07 15:04:43,095.095 | DEBUG    | read_1D_spectrum - io.py:112 : Read 1D spectrum from file:
../../data/test_input_files/FORS_1D.fits
2024-11-07 15:04:43,096.096 | INFO     | read_fits_1D_spectrum - io.py:448 : Attempting to read file:
../../data/test_input_files/FORS_1D.fits
2024-11-07 15:04:43,103.103 | DEBUG    | get_flux_units - io.py:761 : Using BUNIT: 'erg/cm2/s/A' for flux units
2024-11-07 15:04:43,105.105 | WARNING  | _showwarning - logger.py:235 : UnitsWarning: 'erg/cm2/s/A' contains multiple slashes, which is discouraged by the FITS standard
2024-11-07 15:04:43,106.106 | INFO     | get_wavelength_units - io.py:784 : No unit found in header for wavelength (axis 1), assuming 'pix'
2024-11-07 15:04:43,106.106 | DEBUG    | get_wavelength_constructor - io.py:848 : Using the FITS CD matrix.
2024-11-07 15:04:43,106.106 | DEBUG    | get_wavelength_constructor - io.

In [3]:
spec1.info()
spec2.info()
spec3.info()

2024-11-07 15:04:43,193.193 | INFO     | info - spectrum.py:360 : Properties of <zhunter.spectrum.OneDSpectrum object at 0x16356e8f0>:
{'wvlg': {'step': <Quantity [3.2481, 3.2481, 3.2481, ..., 3.2481, 3.2481, 3.2481] pix>,
          'min': <Quantity 2998.5679 pix>,
          'max': <Quantity 9647.3887 pix>,
          'span': <Quantity 6648.8208 pix>},
 'flux': {'q975': <Quantity 0. erg / (A s cm2)>,
          'q025': <Quantity -0. erg / (A s cm2)>},
 'smoothing': {'func': None, 'args': None}}
2024-11-07 15:04:43,194.194 | INFO     | info - spectrum.py:360 : Properties of <zhunter.spectrum.OneDSpectrum object at 0x16356e710>:
{'wvlg': {'step': <Quantity [0.02, 0.02, 0.02, ..., 0.02, 0.02, 0.02] nm>,
          'min': <Quantity 550. nm>,
          'max': <Quantity 700. nm>,
          'span': <Quantity 150. nm>},
 'flux': {'q975': <Quantity 0. erg / (Angstrom s cm2)>,
          'q025': <Quantity 0. erg / (Angstrom s cm2)>},
 'smoothing': {'func': None, 'args': None}}
2024-11-07 15:04:43,19

## Change the units
In case the spectrum was read with wrong units

In [4]:
spec1.set_wvlg_unit(u.AA)
spec1.info()

spec3.set_wvlg_unit(u.AA)
spec3.info()


2024-11-07 15:04:43,201.201 | DEBUG    | set_wvlg_unit - spectrum.py:168 : Setting wavelength units to Angstrom
2024-11-07 15:04:43,202.202 | DEBUG    | _update_units_from_base - spectrum.py:290 : Updating units from base spectrum
2024-11-07 15:04:43,203.203 | INFO     | info - spectrum.py:360 : Properties of <zhunter.spectrum.OneDSpectrum object at 0x16356e8f0>:
{'wvlg': {'step': <Quantity [3.2481, 3.2481, 3.2481, ..., 3.2481, 3.2481, 3.2481] Angstrom>,
          'min': <Quantity 2998.5679 Angstrom>,
          'max': <Quantity 9647.3887 Angstrom>,
          'span': <Quantity 6648.8208 Angstrom>},
 'flux': {'q975': <Quantity 0. erg / (A s cm2)>,
          'q025': <Quantity -0. erg / (A s cm2)>},
 'smoothing': {'func': None, 'args': None}}
2024-11-07 15:04:43,203.203 | DEBUG    | set_wvlg_unit - spectrum.py:168 : Setting wavelength units to Angstrom
2024-11-07 15:04:43,205.205 | DEBUG    | _update_units_from_base - spectrum.py:290 : Updating units from base spectrum
2024-11-07 15:04:43,

## Visualizing the spectrum

In [5]:
# Classic plotting widget
fig_cl = pg.PlotWidget()
vb_cl = fig_cl.plotItem.vb

fig_cl.show()

In [13]:
wvlg, flux, unc = spec1.extract_subspectrum_between()
y_unit = u.Unit('erg/(s cm2 AA)')
convert_flux_with_unc_propagation(flux, unc, to_unit=y_unit)

UnitsError: Argument 'flux' to function 'convert_flux_with_unc_propagation' must be in units convertible to one of: 'erg/s/cm2/AA', 'Jy'.

In [ ]:
from zhunter.conversions import convert_flux_to_value, convert_flux_with_unc_propagation
# y = convert_flux_to_value(flux=flux, unit=y_unit, wvlg=wvlg)


In [8]:
sp_vr = spec1.plot_pyqt(vb=vb_cl)


2024-11-07 15:05:00,912.912 | DEBUG    | update_pyqt - spectrum.py:675 : Updating visual representation of spectrum
2024-11-07 15:05:00,912.912 | DEBUG    | update_pyqt - spectrum.py:686 : No bounds provided, using bounds stored in dictionary: None
2024-11-07 15:05:00,913.913 | DEBUG    | update_pyqt - spectrum.py:699 : No units specified for wavelength, using existing units: Angstrom
2024-11-07 15:05:00,913.913 | DEBUG    | update_pyqt - spectrum.py:710 : No units specified for flux, using existing units: erg / (A s cm2)
2024-11-07 15:05:00,915.915 | DEBUG    | update_pyqt - spectrum.py:734 : Uncertainty spectrum is filled with 0. Not displaying.


UnitsError: Argument 'flux' to function 'convert_flux_to_value' must be in units convertible to one of: 'erg/s/cm2/AA', 'Jy'.

In [7]:
colors = iter(colors["specsys"])
sp_vrs = []
# for spec in (spec1, spec2, spec3):
sp_vr = spec1.plot_pyqt(vb=vb_cl, color=next(colors))
sp_vrs.append(sp_vr)
    # vb_cl.addItem(sp_vr.PlotItem)


2024-11-07 15:04:43,598.598 | DEBUG    | update_pyqt - spectrum.py:675 : Updating visual representation of spectrum
2024-11-07 15:04:43,599.599 | DEBUG    | update_pyqt - spectrum.py:686 : No bounds provided, using bounds stored in dictionary: None
2024-11-07 15:04:43,599.599 | DEBUG    | update_pyqt - spectrum.py:699 : No units specified for wavelength, using existing units: Angstrom
2024-11-07 15:04:43,599.599 | DEBUG    | update_pyqt - spectrum.py:710 : No units specified for flux, using existing units: erg / (A s cm2)
2024-11-07 15:04:43,600.600 | DEBUG    | update_pyqt - spectrum.py:734 : Uncertainty spectrum is filled with 0. Not displaying.


UnitsError: Argument 'flux' to function 'convert_flux_to_value' must be in units convertible to one of: 'erg/s/cm2/AA', 'Jy'.

## Create a second representation of the same spectrum

In [18]:
sp_vr2 = OneDSpectrumVisRep(spec, color=colors['specsys'][1])

2024-11-01 11:58:25,051.051 | INFO     | update - spectrum.py:558 : Updating visual representation of spectrum
2024-11-01 11:58:25,052.052 | DEBUG    | update - spectrum.py:583 : No units specified for wavelength, using existing units: nm
2024-11-01 11:58:25,052.052 | DEBUG    | update - spectrum.py:594 : No units specified for flux, using existing units: 1e-18 erg / (Angstrom s cm2)


In [19]:
# Custom plotting widget
from zhunter.OneDGraphicsWidget import OneDSpectralWidget
fig = OneDSpectralWidget()
fig.set_up_plot()
vb = fig.ax1D.vb
vb.addItem(sp_vr2.PlotItem)
vb.addItem(sp_vr2.PlotItem_unc)
fig.show()

2024-11-01 11:58:47,925.925 | INFO     | set_up_plot - OneDGraphicsWidget.py:64 : Setting up a new plot called '1D'


2024-11-01 11:58:57,031.031 | DEBUG    | keyPressEvent - OneDGraphicsWidget.py:379 : Key: Control, Mouse position: [436,235]


## Smooth the spectrum
Make sure the smoothing is applied to both visual representations

In [20]:
from zhunter.smoothing import convolve_gaussian
spec.apply_smoothing(
    func=convolve_gaussian,
    args={'sigma':10},
    )

2024-11-01 11:59:42,566.566 | DEBUG    | apply_smoothing - spectrum.py:374 : Applying smoothing:
{'args': {'sigma': 10}, 'func': <function convolve_gaussian at 0x13fa29580>}
2024-11-01 11:59:42,567.567 | DEBUG    | convolve_gaussian - smoothing.py:76 : Performing Gaussian convolution
2024-11-01 11:59:42,567.567 | WARNING  | convolve_gaussian - smoothing.py:80 : Error/uncertainty propagation is not implemented for convolution.
2024-11-01 11:59:42,568.568 | DEBUG    | _update_data - spectrum.py:315 : Updating the following data: ['wvlg', 'flux', 'unc']
2024-11-01 11:59:42,569.569 | INFO     | update - spectrum.py:558 : Updating visual representation of spectrum
2024-11-01 11:59:42,569.569 | DEBUG    | update - spectrum.py:583 : No units specified for wavelength, using existing units: nm
2024-11-01 11:59:42,569.569 | DEBUG    | update - spectrum.py:594 : No units specified for flux, using existing units: 1e-18 erg / (Angstrom s cm2)
2024-11-01 11:59:42,570.570 | INFO     | update - spectr

In [ ]:
spec.apply_smoothing(func=None, apply=False)

In [ ]:
# Add another spectrum
spec2 = OneDSpectrum()
wvlg, flux, unc = generate_fake_1D_spectrum(
    flux_scale=1,
    SNR=20,
    emission_line={'amplitude':5},
)

spec2.load_from_data(
    wvlg=wvlg,
    flux=flux,
    unc=unc,
)
sp_vr3 = OneDSpectrumVisRep(spec2, color=colors['specsys'][0])


vb.addItem(sp_vr3.PlotItem)
vb.addItem(sp_vr3.PlotItem_unc)


2024-11-01 16:27:39,778.778 | DEBUG    | generate_fake_1D_spectrum - misc.py:104 : Adding an emission line with mean: 656.28, stddev: 1, amplitude: 5
2024-11-01 16:27:39,782.782 | DEBUG    | _reset_data - spectrum.py:298 : Resetting data to base spectrum
2024-11-01 16:27:39,784.784 | INFO     | update - spectrum.py:558 : Updating visual representation of spectrum
2024-11-01 16:27:39,784.784 | DEBUG    | update - spectrum.py:583 : No units specified for wavelength, using existing units: nm
2024-11-01 16:27:39,785.785 | DEBUG    | update - spectrum.py:594 : No units specified for flux, using existing units: erg / (Angstrom s cm2)


2024-11-01 16:27:58,758.758 | DEBUG    | keyPressEvent - OneDGraphicsWidget.py:379 : Key: Control, Mouse position: [380,338]
